# Gamry DTA → spec-echem TXT conversion

Standard post-collection step: convert the Gamry `.DTA` files for a run into the clean
tab-separated `.txt` files (`CV.txt`, `steps(N).txt`, `dedoping(N).txt`) that the analysis
code and the spec-echem GUI expect, written **into the same run folder as the spectra**.

**Requires** the `gamry_parser` package (`pip install gamry-parser`) to read raw `.DTA`.
This dependency is only needed here on the instrument machine, not by the spec-echem GUI.

**Usage:** edit the two `dta_path` / `txt_path` lines in the driver cell for your run, then run.
Naming: `CV` → `CV.txt`; `prededope` is skipped; `{name}_#N.dta` → `{name}(N-1).txt`.

_Note: the conversion logic will eventually move into `spec_echem/gamry_data.py`; for now this
notebook is the standard tool. See repo TODO for the planned own-parser / consistency cleanups._

In [ ]:
"""
Created as a class on Sunday, 10-31-2021

@author: Dean Waldow

"""

import gamry_parser as parser
import random
import pandas as pd
import numpy as np
from pathlib import Path
import os
from datetime import datetime
import time

class ConvertDTA():
    '''#DTA file type conversion'''
    def __init__(self, dta_path = None, txt_path = None, filename = None, exp_type = None):
        self.dta_path = dta_path
        self.txt_path = txt_path
        self.filename = filename
        self.exp_type = exp_type
        return
    
    def dta_path_info(self):
        print (self.dta_path)
        return
    
    def txt_path_info(self):
        print (self.txt_path)
        return
    
    def get_file_type(self):
        gp = parser.GamryParser()
        full_path = Path(self.dta_path) / self.filename
        gp.load(filename = full_path)
        self.exp_type = gp.get_experiment_type()
        print (self.exp_type)
        return
        
    def save_file(self):
        '''Heart of this class is to save the'''
        print ("At file type decision point! Type: ", self.exp_type)
        dta_full_path = Path(self.dta_path) / self.filename
        txt_full_path = dta_full_path.with_suffix('.txt')
        print ("Full text path-filename: ",txt_full_path)
        txt_filename =  os.path.basename(txt_full_path )
        print ("Text filename: ",txt_filename)
        output_path = Path(self.txt_path)
        print ("Output_path: ", output_path)
        print ("Output txt_full_path=: ", txt_full_path)
        
        if self.exp_type == "CHRONOA":
            print ("Rewriting the chrono data.")
            ca = parser.ChronoAmperometry(to_timestamp=True)
            #ca = parser.ChronoAmperometry(to_timestamp=False)
            ca.load(filename=dta_full_path)
            echem = ca.get_curve_data()
            print (echem)
            echem['DataIndex'] = range(0, len(echem))
            echem.insert(1, "Corrected", echem["T"])
            corrected = echem["Corrected"] - echem.iloc[0, 0]
            echem["Corrected"]=corrected.dt.total_seconds()
            test_time = echem["T"] - echem.iloc[0, 0]
            echem["T"] = test_time.dt.total_seconds() + 100
            #
            #Time (s)	Corrected time (s)	WE(1).Potential (V)	WE(1).Current (A)	Index
            #
            echem.columns = ["Time (s)","Corrected time (s)","WE(1).Potential (V)","WE(1).Current (A)","Index"]
            echem.to_csv(txt_full_path, index=False, sep ='\t')
            #output_df_all.to_csv(r'c:\Users\inst-chem\Downloads\output_df_all.csv', header=False, index=False, sep ='\t')

        elif self.exp_type =="CV":
            print ("Rewriting CV file type!")

            gp = parser.GamryParser(to_timestamp=True)
            gp.load(filename=dta_full_path)

            curves = gp.get_curve_count()

            for curve_index in range(curves):
                print("Curved Index: ",curve_index)
                print(gp.get_curve_data(curve_index))
                temp_gp = gp.get_curve_data(curve_index)
                temp_gp = temp_gp.drop(temp_gp.columns[[0,3,4,5,6,7,8]], axis=1)
                temp_gp.columns = ['WE(1).Potential (V)','WE(1).Current (A)']
                print (temp_gp)
                if curve_index == 0:
                    all_gp = temp_gp
                    #print (all_gp)
                elif curve_index > 0:
                    all_gp = pd.concat([all_gp,temp_gp], axis=0, ignore_index=True)
                    #print (all_gp)
                temp_gp = 0
 
            all_gp.to_csv(txt_full_path, index=False, sep ='\t')
        else:
            print ("No expected type Found. Failed to convert.")
        return
    

class ConvertDTAtoTXT():
    '''#DTA file type conversion'''
    def __init__(self, dta_path = None, txt_path = None, dta_filename = None, txt_filename = None, exp_type = None):
        self.dta_path = dta_path
        self.txt_path = txt_path
        self.dta_filename = dta_filename
        self.txt_filename = txt_filename
        self.exp_type = exp_type
        return
    
    def dta_path_info(self):
        print (self.dta_path)
        return
    
    def txt_path_info(self):
        print (self.txt_path)
        return
    
    def get_file_type(self):
        gp = parser.GamryParser()
        full_path = Path(self.dta_path) / self.dta_filename
        gp.load(filename = full_path)
        self.exp_type = gp.get_experiment_type()
        print (self.exp_type)
        return
        
    def save_file(self):
        '''Heart of this class is to save the'''
        print ("At file type decision point! Type: ", self.exp_type)
        dta_full_path = Path(self.dta_path) / self.dta_filename
        txt_full_path = Path(self.txt_path) / self.txt_filename
        print ("Full DTA path-filename: ",dta_full_path)
        print ("Full TXT path-filename: ",txt_full_path)
        
        if self.exp_type == "CHRONOA":
            print ("Rewriting the chrono data.")
            ca = parser.ChronoAmperometry(to_timestamp=True)
            #ca = parser.ChronoAmperometry(to_timestamp=False)
            ca.load(filename=dta_full_path)
            echem = ca.get_curve_data()
            print (echem)
            echem['DataIndex'] = range(0, len(echem))
            echem.insert(1, "Corrected", echem["T"])
            corrected = echem["Corrected"] - echem.iloc[0, 0]
            echem["Corrected"]=corrected.dt.total_seconds()
            test_time = echem["T"] - echem.iloc[0, 0]
            echem["T"] = test_time.dt.total_seconds() + 100
            #
            #Time (s)	Corrected time (s)	WE(1).Potential (V)	WE(1).Current (A)	Index
            #
            echem.columns = ["Time (s)","Corrected time (s)","WE(1).Potential (V)","WE(1).Current (A)","Index"]
            echem.to_csv(txt_full_path, index=False, sep ='\t')
            #output_df_all.to_csv(r'c:\Users\inst-chem\Downloads\output_df_all.csv', header=False, index=False, sep ='\t')

        elif self.exp_type =="CV":
            print ("Rewriting CV file type!")

            gp = parser.GamryParser(to_timestamp=True)
            gp.load(filename=dta_full_path)

            curves = gp.get_curve_count()

            for curve_index in range(curves):
                print("Curved Index: ",curve_index)
                #print(gp.get_curve_data(curve_index))
                temp_gp = gp.get_curve_data(curve_index)
                temp_gp = temp_gp.drop(temp_gp.columns[[0,3,4,5,6,7,8]], axis=1)
                temp_gp.columns = ['WE(1).Potential (V)','WE(1).Current (A)']
                #print (temp_gp)
                if curve_index == 0:
                    all_gp = temp_gp
                    #print (all_gp)
                elif curve_index > 0:
                    all_gp = pd.concat([all_gp,temp_gp], axis=0, ignore_index=True)
                    #print (all_gp)
                temp_gp = 0
 
            all_gp.to_csv(txt_full_path, index=False, sep ='\t')
        else:
            print ("No expected type Found. Failed to convert.")
        return

## Batch-convert a run folder
Edit the two paths below, then run.

In [ ]:
dta_path = Path(r'C:\Users\Public\Documents\My Gamry Data\20250723_P3HT9010rerun_KPF6')
txt_path = Path(r'C:\Users\inst-chem\Documents\specechem_data\20250723_P3HT9010rerun_KPF6')

filelist = [f for f in os.listdir(dta_path) if not f.startswith('.')]

for name in filelist:
    dta_full_path = dta_path / name
    base_filename = os.path.basename(dta_full_path).split('.')[0]
    print (base_filename)
    if base_filename == 'CV':
        print (".....convert CV")
        new_name = "CV.txt"
        print (name, " ---> ",new_name)
        cd = ConvertDTAtoTXT(dta_path,txt_path, name, new_name)
        cd.get_file_type()
        cd.save_file()
    elif base_filename == 'prededope':
        print (f".....skipping: {base_filename}")
    else:
        print ("..... OLD NAME: ", name)
        part1 = base_filename.split('_#')[0]
        part2 = base_filename.split('_#')[1]
        part2b = str((int(part2)-1))
        new_name = part1+"("+part2b+")"+".txt"
        print (name, " ---> ",new_name)
        cd = ConvertDTAtoTXT(dta_path,txt_path, name, new_name)
        cd.get_file_type()
        cd.save_file()
